# Macroeconomic Regime Switching via Hidden Markov Models
In this notebook, we explore the latent (hidden) regimes in the macroeconomic data using Gaussian HMMs. The hypothesis is that different regimes (e.g. expansionary vs. recessionary) structurally change the correlation of alternative data features to future returns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

plt.style.use('dark_background')
sns.set_palette('muted')

### 1. Load the Raw Macro Data
First, we load the raw data downloaded from FRED (Federal Reserve Economic Data).

In [ ]:
repo_root = Path.cwd().parent
macro_path = repo_root / "data" / "raw" / "macro_data.parquet"

try:
    df_macro = pd.read_parquet(macro_path)
    df_macro['date'] = pd.to_datetime(df_macro['date'])
    df_macro = df_macro.sort_values('date').set_index('date')
    print(f"Macro data loaded with shape: {df_macro.shape}")
except Exception as e:
    print(f"Data not found. Please run the ingestion pipeline first. Error: {e}")

In [ ]:
df_macro.head()

### 2. Fit the Hidden Markov Model (HMM)
We select core inflationary and growth indicators to identify regimes.

In [ ]:
features = ['fed_funds_rate', 'cpi', 'unemployment_rate', 'wti_oil']
df_hmm = df_macro[features].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_hmm.values)

model = hmm.GaussianHMM(n_components=2, covariance_type="full", n_iter=100, random_state=42)
model.fit(X_scaled)

df_hmm['regime'] = model.predict(X_scaled)
probs = model.predict_proba(X_scaled)
df_hmm['prob_regime_0'] = probs[:, 0]
df_hmm['prob_regime_1'] = probs[:, 1]

### 3. Visualize the Regimes
Let's plot the Federal Funds Rate and shade the background according to the dominant HMM regime.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.plot(df_hmm.index, df_hmm['fed_funds_rate'], color='cyan', label='Fed Funds Rate')

ax.fill_between(df_hmm.index, ax.get_ylim()[0], ax.get_ylim()[1], 
                where=(df_hmm['regime'] == 1), color='red', alpha=0.3, label='Regime 1 (High Rate/Inflation)')
ax.fill_between(df_hmm.index, ax.get_ylim()[0], ax.get_ylim()[1], 
                where=(df_hmm['regime'] == 0), color='green', alpha=0.3, label='Regime 0 (Low Rate/Growth)')

ax.set_title('Macroeconomic Regimes Detected by Gaussian HMM', fontsize=16)
ax.legend()
plt.show()